# AgentOps Getting Started Guide

This notebook walks through a complete AgentOps lifecycle: configure credentials, create an agent definition, deploy it, validate the running agent, and clean up.

## Why This Matters
Building an AI agent is only half the job — the hard part is running it reliably for real users. AgentOps is the part of Teradata AgentStack that takes an agent from source code to a secured, running service and gives you one consistent way to operate it.

- **From prototype to production.** Package an agent once (as a ZIP archive or a Git repository) and deploy it as a managed, always-on service — with no bespoke infrastructure to wire up.
- **Runs next to your data.** Agents are deployed inside the Teradata platform, close to the data they reason over, instead of shipping data out to a separate stack.
- **Secure by default.** API keys and registry credentials live in managed secrets, so nothing sensitive is baked into your code or container images.
- **Built to evolve.** Versioning, upgrades, and clean teardown let you ship new agent versions and roll forward safely without losing track of what is running.

## What You Will Accomplish
By the end of this notebook you will have:
1. Configured credentials and reviewed the demo settings.
2. Inspected the available SDK operations with `blueprint()`.
3. Created an agent definition from a ZIP archive or a Git repository.
4. Deployed the agent and waited for it to become active.
5. Validated the agent (`/health`, `/info`, `/invoke`).
6. Cleaned up every resource you created.

---

## Table of Contents

### [1. Setup](#1-setup)
- 1.1 Import required modules
- 1.2 Authentication
- 1.3 Provide connection details
- 1.4 Demo configuration *(shared settings)*
- 1.5 Initialize the SDK clients

### [2. Inspect Blueprint](#2-inspect-blueprint)

### [3. Health Check](#3-health-check)

### [4. Create an Agent Definition](#4-create-an-agent-definition)
- 4.1 Create from a ZIP archive *(default path)*
- 4.2 Create from Git *(optional)*
- 4.3 List agent definitions
- 4.4 Get agent definition details

### [5. Deploy the Agent](#5-deploy-the-agent)
- 5.1 Create a runtime secret *(optional)*
- 5.2 Create an image pull secret *(optional)*
- 5.3 Verify secrets in the workspace
- 5.4 Create the deployment
- 5.5 Wait for the deployment to become active
- 5.6 Get deployment details
- 5.7 List deployments
- 5.8 Get deployment logs

### [6. Validate the Agent](#6-validate-the-agent)
- 6.1 Get agent URL
- 6.2 Test agent endpoints
- 6.3 Restart the deployment *(optional)*

### [7. Version Management *(optional)*](#7-version-management-optional)
- 7.1 Update agent definition metadata
- 7.2 Pull the latest Git version *(optional)*
- 7.3 Get version details
- 7.4 List versions
- 7.5 Upgrade the deployment *(optional)*
- 7.6 Monitor upgrade progress

### [8. Cleanup](#8-cleanup)
- 8.1 Delete the deployment
- 8.2 Delete secrets
- 8.3 Archive an agent definition version *(optional)*
- 8.4 Archive the agent definition

### [9. Data Models Reference](#9-data-models-reference)


<a id="setup"></a>
## 1. Setup

### 1.1 Import Required Modules

Imports the public SDK classes and models, and defines a small `show_output()` helper so responses are easy to read.


In [ ]:
import json
import time
from getpass import getpass
from pprint import pprint

import requests

from teradata_agentstack import BearerAuth
from teradata_agentstack.agentops import (
    AgentOpsClient,
    Definitions,
    Deployments,
    Framework,
    Health,
    SecretMountType,
    SecretType,
    Secrets,
    blueprint,
)
from teradata_agentstack.agentops.models import (
    AgentDefinitionCreateRequest,
    AgentDefinitionUpdateRequest,
    AgentDefinitionVersionArchiveRequest,
    ArtifactsConfig,
    DeploymentCreateRequest,
    DeploymentEngineTypeConfig,
    DeploymentResourceConfig,
    DeploymentServiceConfig,
    DeploymentUpdateRequest,
    DockerConfigJsonSecretData,
    GitConnectionValidateRequest,
    GitCredentials,
    GitSyncRequest,
    OpaqueSecretData,
    SecretCreateRequest,
    SecretDataConfig,
    SecretMountConfig,
)


def show_output(label, value):
    """Pretty-print an SDK response (Pydantic model or plain value)."""
    print(f"\n{label}:")
    formatter = getattr(value, "model_dump", None) or getattr(value, "dict", None) or (lambda: value)
    pprint(formatter() if callable(formatter) else formatter, sort_dicts=False, width=120)

<a id="authentication"></a>
### 1.2 Authentication

The `teradata_agentstack` AgentOps client supports **4 authentication modes** and **3 ways to provide credentials**.

The credential sources are resolved in this order: direct `auth` parameter, environment variables, then YAML config file.

#### Authentication Modes

| Auth Mode | Class | Required Fields |
| --- | --- | --- |
| Bearer Token | `BearerAuth` | `auth_bearer` |
| Basic Auth | `BasicAuth` | `username`, `password` |
| Client Credentials (OAuth2) | `ClientCredentialsAuth` | `auth_token_url`, `auth_client_id`, `auth_client_secret` |
| Device Code (OAuth2) | `DeviceCodeAuth` | `auth_token_url`, `auth_client_id`, `auth_client_secret`, `auth_device_auth_url` |

This notebook uses `BearerAuth` for AgentOps requests. You enter the token securely via `getpass` in step 1.3 — it is never hardcoded.

### 1.3 Provide Connection Details

Enter the AgentOps service URL when prompted &mdash; usually the only value you need to type. This cell also captures:

- `AUTH_TOKEN` &mdash; entered securely via `getpass` so it is never echoed or stored in the notebook.
- `OPENAI_API_KEY` &mdash; optional; only needed if your agent calls OpenAI at runtime. Press Enter to skip it.
- `SSL_VERIFY` &mdash; defaults to `False` for demo environments with self-signed certificates. Set `True` when your environment uses trusted certificates.

No credential values are printed &mdash; only whether each one was captured.

In [ ]:
# Endpoint for the AgentOps service you are targeting.
BASE_URL = getpass("Enter BASE_URL: ").strip()

# Bearer token (JWT) for the AgentOps service, entered securely.
AUTH_TOKEN = getpass("Enter AUTH_TOKEN: ").strip()

# Optional: only needed if your agent calls OpenAI at runtime. Press Enter to skip.
OPENAI_API_KEY = getpass("Enter OPENAI_API_KEY (optional, press Enter to skip): ").strip()

# Verify the endpoint's TLS certificate? Defaults to False for demo environments
# that use self-signed certificates. Set True when your environment uses trusted certs.
SSL_VERIFY = False

print("Connection details captured.")
print(f"BASE_URL configured:   {bool(BASE_URL)}")
print(f"AUTH_TOKEN configured: {bool(AUTH_TOKEN)}")
print(f"OPENAI_API_KEY set:    {bool(OPENAI_API_KEY)}")
print(f"SSL_VERIFY:            {SSL_VERIFY}")

### 1.4 Demo Configuration

Holds the settings shared across several cells: the workspace, the agent's identity (name, description, framework, tags), the Git source, and the runtime secret name — plus the state variables the notebook fills in as it runs. Step-specific values (ZIP path, deployment settings, secret and registry names, version identifiers) now sit at the top of the cell that uses them, so each step reads top-to-bottom on its own.

> **Run this cell once, near the start.** It resets the state variables (`AGENT_DEF_ID`, `DEPLOYMENT_ID`, secret IDs, `agent_url`) to `None`, so re-running it midway will clear the IDs the notebook has already captured.


In [129]:
# ── Workspace ───────────────────────────────────────────────────────────────
WORKSPACE_ID = None  # None = default workspace

# ── Agent identity (shared by the create, list, and update cells) ───────────
AGENT_DEFINITION_NAME = "langgraph-agent-demo-new-test"
AGENT_DEFINITION_DESCRIPTION = "LangGraph-based conversational agent"
AGENT_FRAMEWORK = Framework.langgraph
AGENT_TAGS = ["langgraph", "conversational-ai", "memory", "postgresql"]

# ── Git source (shared by the optional Git create and pull cells) ───────────
GIT_REPO_URL = "https://github.com/<org>/<repo>.git"
GIT_BRANCH = "main"
GIT_TARGET_PATH = "examples/agents/sample_agent"

# ── Runtime secret name (shared by the secret-create and deployment cells) ──
RUNTIME_SECRET_NAME = "agent-runtime-config-demo-new-test"

# ── Mutable state populated as the notebook runs ────────────────────────────
AGENT_DEF_ID = None
DEPLOYMENT_ID = None
SECRET_ID_1 = None
SECRET_ID_2 = None
agent_url = None

print("Demo configuration loaded.")


Demo configuration loaded.


### 1.5 Initialize the SDK Clients

Creates the authenticated `AgentOpsClient` and the four API objects — `Definitions`, `Deployments`, `Secrets`, and `Health` — used throughout the notebook.


In [130]:
auth = BearerAuth(auth_bearer=AUTH_TOKEN)
client = AgentOpsClient(base_url=BASE_URL, auth=auth, ssl_verify=SSL_VERIFY)

agent_definitions = Definitions(client=client)
deployments = Deployments(client=client)
secrets = Secrets(client=client)
health_api = Health(client=client)

print("AgentOps SDK clients initialized.")


AgentOps SDK clients initialized.


<a id="blueprint"></a>
## 2. Inspect Blueprint

Prints a summary of all available AgentOps API groups and operations. Run this once to see what is available before you start creating resources.

In [131]:
blueprint()

----------------------------------------------------------------
Available classes for AgentOps SDK:
    * teradata_agentstack.agentops.Definitions
    * teradata_agentstack.agentops.Deployments
    * teradata_agentstack.agentops.Secrets
    * teradata_agentstack.agentops.Health
----------------------------------------------------------------


<a id="health-check"></a>
## 3. Health Check

Verifies that the AgentOps service is reachable and your credentials are accepted before you create or manage any resources.

In [132]:
health = health_api.health_check()
show_output("Health Check", health)


Health Check:
{'status': 'healthy',
 'service': 'agentops-core',
 'checks': {'api': 'healthy', 'database': 'healthy', 'service_agent': 'healthy'}}


<a id="agent-definitions"></a>
## 4. Create an Agent Definition

An agent definition registers your agent's code, framework, and configuration with the service. You can create one from a local ZIP archive or directly from a Git repository.


### 4.1 Create an Agent Definition from a ZIP Archive

Registers a new agent definition from the local ZIP at `ZIP_PATH`. The request is built entirely from typed SDK models (`AgentDefinitionCreateRequest` + `ArtifactsConfig`), and `poll()` waits until the artifacts finish processing. This is the default path — run this cell to create your definition.


In [ ]:
# Local ZIP archive containing the agent code.
ZIP_PATH = "../data/langgraph_agent.zip"

create_request = AgentDefinitionCreateRequest(
    name=AGENT_DEFINITION_NAME,
    description=AGENT_DEFINITION_DESCRIPTION,
    framework=AGENT_FRAMEWORK,
    artifacts=ArtifactsConfig(storage_type="zip"),
    tags=AGENT_TAGS,
    framework_template="passthrough",
    workspace_id=WORKSPACE_ID,
)

created = agent_definitions.create(body=create_request, file=ZIP_PATH)
show_output("Created Agent Definition", created)

AGENT_DEF_ID = getattr(created, "id", None)
agent_definitions.poll(id=AGENT_DEF_ID)
print(f"AGENT_DEF_ID: {AGENT_DEF_ID}")


### 4.2 Create an Agent Definition from Git *(optional)*

Use this instead of the ZIP path if you'd rather pull the agent straight from a Git repository. Run this cell only when you want the Git option — it prompts for your Git username and personal access token, checks that the repository is reachable, and then creates the definition. The repository URL and branch come from the Demo Configuration cell.


In [ ]:
git_username = getpass("Enter Git username: ").strip()
git_pat = getpass("Enter Git personal access token: ").strip()
git_credentials = GitCredentials(username=git_username, personal_access_token=git_pat)

# 1) Validate the repository is reachable before importing it.
validation = agent_definitions.validate_git_connection(
    body=GitConnectionValidateRequest(
        repository_url=GIT_REPO_URL,
        branch=GIT_BRANCH,
        target_path=GIT_TARGET_PATH or None,
        credentials=git_credentials,
    )
)
show_output("Validate Git Connection", validation)

# 2) Create the agent definition from the validated repository.
created = agent_definitions.create(
    body=AgentDefinitionCreateRequest(
        name=AGENT_DEFINITION_NAME,
        description=AGENT_DEFINITION_DESCRIPTION,
        framework=AGENT_FRAMEWORK,
        artifacts=ArtifactsConfig(
            storage_type="git",
            git_config={
                "repository_url": GIT_REPO_URL,
                "branch": GIT_BRANCH,
                "target_path": GIT_TARGET_PATH or None,
                "credentials": git_credentials,
            },
        ),
        tags=AGENT_TAGS + ["git-import"],
        framework_template="passthrough",
        workspace_id=WORKSPACE_ID,
    )
)
show_output("Created Agent Definition (Git)", created)
AGENT_DEF_ID = getattr(created, "id", None)
agent_definitions.poll(id=AGENT_DEF_ID)
print(f"AGENT_DEF_ID: {AGENT_DEF_ID}")


### 4.3 List Agent Definitions

Lists all agent definitions in the selected workspace so you can confirm the one created above.


In [ ]:
definitions_response = agent_definitions.list(
    workspace_id=WORKSPACE_ID,
    framework=AGENT_FRAMEWORK,
    name=None,
    description=None,
    include_archived=False,
    include_failed=False,
)
show_output("List Agent Definitions", definitions_response)


### 4.4 Get Agent Definition Details

Retrieves the full metadata and current status of the active agent definition before deploying it.


In [135]:
show_output("Get Agent Definition", agent_definitions.get(id=AGENT_DEF_ID))


Get Agent Definition:
{'id': '<uuid>',
 'name': 'langgraph-agent-demo-new-test',
 'description': 'LangGraph-based conversational agent',
 'framework': 'langgraph',
 'status': 'published',
 'tags': ['langgraph', 'conversational-ai', 'memory', 'postgresql', 'git-import'],
 'shared': False,
 'workspace': {'id': '<uuid>', 'name': 'default'},
 'version_count': 1,
 'latest_version': {'id': '<uuid>',
                    'agent_definition_id': '<uuid>',
                    'version_number': 1,
                    'status': 'published',
                    'version_notes': None,
                    'artifacts_path': '<uuid>/agent-artifacts/<uuid>/v1',
                    'error_message': None,
                    'created_at': '2026-06-18T14:37:43.234291Z',
                    'updated_at': '2026-06-18T14:37:45.139125Z'},
 'active_deployment_count': 0,
 'storage_type': 'git',
 'repository_url': 'https://github.com/<org>/<repo>.git',
 'created_at': '2026-06-18T14:37:43.218011Z',
 'updated_at': 

<a id="deployments"></a>
## 5. Deploy the Agent

A deployment is a running instance of an agent definition on Kubernetes. The secret steps (5.1, 5.2) are optional and off by default — the deployment in 5.4 works without them.


### 5.1 Create a Runtime Secret *(optional)*

Run this cell only if your agent needs an extra secret (e.g. an API key) injected as an environment variable at runtime. It prompts for the secret key name (press Enter for the `OPIK_API_KEY` default) and its value, then creates an opaque secret that the deployment cell (5.4) mounts automatically. The deployment works fine without it.


In [136]:
# Prompt for the secret's key name (press Enter to use the default) and its value.
runtime_secret_key = getpass("Enter runtime secret key (default OPIK_API_KEY): ").strip() or "OPIK_API_KEY"
runtime_secret_value = getpass(f"Enter value for {runtime_secret_key}: ").strip()

created_secret = secrets.create(
    workspace_id=WORKSPACE_ID,
    body=SecretCreateRequest(
        name=RUNTIME_SECRET_NAME,
        description="Runtime configuration secrets for the agent deployment",
        secret_data=SecretDataConfig(
            secret_type=SecretType.opaque,
            opaque_data=OpaqueSecretData(data={runtime_secret_key: runtime_secret_value}),
        ),
    ),
)
show_output("Created Opaque Secret", created_secret)
SECRET_ID_1 = getattr(created_secret, "id", None)
print(f"SECRET_ID_1: {SECRET_ID_1}")



Created Opaque Secret:
{'id': '<uuid>',
 'workspace_id': '<uuid>',
 'name': 'agent-runtime-config-demo-new-test',
 'k8s_secret_name': '<redacted-secret>',
 'namespace': 'agentruntime',
 'secret_type': '<redacted-secret>',
 'keys': ['OPIK_API_KEY'],
 'description': 'Runtime configuration secrets for the agent deployment',
 'status': 'active',
 'used_by_deployments': None,
 'created_at': '2026-06-18T14:38:10.824542',
 'created_by': '<user>',
 'updated_at': '2026-06-18T14:38:10.824546',
 'last_rotated_at': None}
SECRET_ID_1: <uuid>


### 5.2 Create an Image Pull Secret *(optional)*

Run this cell only if your `DOCKER_IMAGE` lives in a private registry. The secret name and registry server are set at the top of the cell; it then prompts for the registry username and password and creates a Docker registry secret.


In [137]:
# Name for the image pull secret and the private registry it authenticates to.
IMAGE_PULL_SECRET_NAME = "image-pull-secret-1-demo-new-test"
DOCKER_REGISTRY_SERVER = "https://<registry-host>"

registry_username = getpass("Enter registry username: ").strip()
registry_password = getpass("Enter registry password: ").strip()

created_docker_secret = secrets.create(
    workspace_id=WORKSPACE_ID,
    body=SecretCreateRequest(
        name=IMAGE_PULL_SECRET_NAME,
        description="Authentication for pulling private images",
        secret_data=SecretDataConfig(
            secret_type=SecretType.dockerconfigjson,
            dockerconfigjson_data=DockerConfigJsonSecretData(
                docker_server=DOCKER_REGISTRY_SERVER,
                docker_username=registry_username,
                docker_password=registry_password,
            ),
        ),
    ),
)
show_output("Created Image Pull Secret", created_docker_secret)
SECRET_ID_2 = getattr(created_docker_secret, "id", None)
print(f"SECRET_ID_2: {SECRET_ID_2}")



Created Image Pull Secret:
{'id': '<uuid>',
 'workspace_id': '<uuid>',
 'name': 'image-pull-secret-1-demo-new-test',
 'k8s_secret_name': '<redacted-secret>',
 'namespace': 'agentruntime',
 'secret_type': '<redacted-secret>',
 'keys': ['.dockerconfigjson'],
 'description': 'Authentication for pulling private images',
 'status': 'active',
 'used_by_deployments': None,
 'created_at': '2026-06-18T14:38:30.369169',
 'created_by': '<user>',
 'updated_at': '2026-06-18T14:38:30.369174',
 'last_rotated_at': None}
SECRET_ID_2: <uuid>


### 5.3 Verify Secrets in the Workspace

Lists all secrets visible to the selected workspace. Confirm both secrets appear here before creating the deployment.

In [ ]:
show_output(
    "List Secrets",
    secrets.list(workspace_id=WORKSPACE_ID, offset=0, limit=50),
)


### 5.4 Create the Deployment

Deploys the active agent definition as a running Kubernetes workload. The deployment settings (name, namespace, image, resources, runtime env) are defined at the top of this cell, and the whole request is built from typed SDK models. `OPENAI_API_KEY` (if set) and the optional runtime secret are wired in automatically.


In [139]:
# ── Deployment settings ─────────────────────────────────────────────────────
DEPLOYMENT_NAME = "LangGraph-Agent-test-2-new-test"
DEPLOYMENT_NAMESPACE = "agentruntime"
DEPLOYMENT_ENVIRONMENT = "dev"
DOCKER_IMAGE = "<registry-host>/<image>:<tag>"
ENGINE_TYPE = "DOCKER_RESTFUL"
REPLICAS = 1
NODE_TYPE = "default-wl"
MEMORY = "2Gi"
CPU = "1000m"
SERVICE_PORT = 5000

# Values passed to the agent container.
RUNTIME_CONFIG = {
    "ENFORCE_AUTHORIZATION": "true",
    "OPIK_CONSOLE_LOGGING_LEVEL": "DEBUG",
}
if OPENAI_API_KEY:
    RUNTIME_CONFIG["OPENAI_API_KEY"] = OPENAI_API_KEY

# Mount the runtime secret created earlier, if any.
runtime_secrets_config = None
if SECRET_ID_1:
    runtime_secrets_config = [SecretMountConfig(secret_name=RUNTIME_SECRET_NAME, mount_type=SecretMountType.env)]

created_dep = deployments.create(
    body=DeploymentCreateRequest(
        agent_definition_id=AGENT_DEF_ID,
        deployment_name=DEPLOYMENT_NAME,
        namespace=DEPLOYMENT_NAMESPACE,
        environment=DEPLOYMENT_ENVIRONMENT,
        version_identifier=None,
        engine_type=ENGINE_TYPE,
        enable_memory=True,
        in_house_observability=True,
        runtime_config=RUNTIME_CONFIG,
        runtime_secrets_config=runtime_secrets_config,
        engine_type_config=DeploymentEngineTypeConfig(
            docker_image=DOCKER_IMAGE,
            replicas=REPLICAS,
            node_type=NODE_TYPE,
            resources=DeploymentResourceConfig(memory=MEMORY, cpu=CPU),
            service=DeploymentServiceConfig(type="ClusterIP", port=SERVICE_PORT, target_port=SERVICE_PORT),
        ),
    )
)
show_output("Created Deployment", created_dep)
DEPLOYMENT_ID = getattr(created_dep, "agent_id", None) or getattr(created_dep, "id", None)
print(f"DEPLOYMENT_ID: {DEPLOYMENT_ID}")


Created Deployment:
{'in_house_observability': True,
 'agent_id': '<uuid>',
 'agent_definition_id': '<uuid>',
 'agent_definition_name': 'langgraph-agent-demo-new-test',
 'workspace_id': '<uuid>',
 'deployment_name': 'LangGraph-Agent-test-2-new-test',
 'environment': 'dev',
 'namespace': 'agentruntime',
 'version_id': '<uuid>',
 'version_number': 1,
 'deployment_history': [{'user': '<user>',
                         'record_id': '<uuid>',
                         'timestamp': '2026-06-18T14:38:47.911245+00:00',
                         'version_id': '<uuid>',
                         'previous_version_id': None}],
 'runtime_config': {'OPENAI_API_KEY': '<redacted-secret>',
                    'OPIK_PROJECT_NAME': '<uuid>',
                    'OPIK_URL_OVERRIDE': 'http://<internal-host>:8080',
                    'ENFORCE_AUTHORIZATION': 'true',
                    'OPIK_CONSOLE_LOGGING_LEVEL': 'DEBUG'},
 'enable_memory': True,
 'resources': {'cpu': '1000m',
               'engine': 'py

### 5.5 Wait for the Deployment to Become Active

Polls the deployment until it reaches a terminal state: `active`, `failed`, or `stopped`.


In [140]:
deployments.poll(id=DEPLOYMENT_ID)

Polling Deployments (every 10s, max 30 attempts)...
  [1/30] Status: deploying
    current_step: Creating Kubernetes resources
    steps_completed: pending, building, pushing
    steps_remaining: health_check, active
  [2/30] Status: active
    current_step: Deployment active and healthy
    steps_completed: pending, building, pushing, deploying, health_check, active
  Done — status: active


DeploymentResponse(in_house_observability=True, agent_id='<uuid>', agent_definition_id='<uuid>', agent_definition_name='langgraph-agent-demo-new-test', workspace_id='<uuid>', deployment_name='LangGraph-Agent-test-2-new-test', environment='dev', namespace='agentruntime', version_id='<uuid>', version_number=1, deployment_history=[{'user': '<user>', 'record_id': '<uuid>', 'timestamp': '2026-06-18T14:38:47.911245+00:00', 'version_id': '<uuid>', 'previous_version_id': None}], runtime_config={'OPENAI_API_KEY': '<redacted-secret>', 'OPIK_PROJECT_NAME': '<uuid>', 'OPIK_URL_OVERRIDE': 'http://<internal-host>:8080', 'ENFORCE_AUTHORIZATION': 'true', 'OPIK_CONSOLE_LOGGING_LEVEL': 'DEBUG'}, enable_memory=True, resources={'cpu': '1000m', 'engine': 'python', 'memory': '2Gi', 'service': {'port': 5000, 'type': 'ClusterIP', 'target_port': 5000}, 'replicas': 1, 'node_type': 'default-wl', 'docker_image': '<registry-host>/<image>:<tag>', 'image_pull_secrets': None}, status='active', current_step='Deploymen

### 5.6 Get Deployment Details

Retrieves the current deployment status, runtime settings, and endpoint information.

In [141]:
show_output("Deployment Details", deployments.get(id=DEPLOYMENT_ID))


Deployment Details:
{'in_house_observability': True,
 'agent_id': '<uuid>',
 'agent_definition_id': '<uuid>',
 'agent_definition_name': 'langgraph-agent-demo-new-test',
 'workspace_id': '<uuid>',
 'deployment_name': 'LangGraph-Agent-test-2-new-test',
 'environment': 'dev',
 'namespace': 'agentruntime',
 'version_id': '<uuid>',
 'version_number': 1,
 'deployment_history': [{'user': '<user>',
                         'record_id': '<uuid>',
                         'timestamp': '2026-06-18T14:38:47.911245+00:00',
                         'version_id': '<uuid>',
                         'previous_version_id': None}],
 'runtime_config': {'OPENAI_API_KEY': '<redacted-secret>',
                    'OPIK_PROJECT_NAME': '<uuid>',
                    'OPIK_URL_OVERRIDE': 'http://<internal-host>:8080',
                    'ENFORCE_AUTHORIZATION': 'true',
                    'OPIK_CONSOLE_LOGGING_LEVEL': 'DEBUG'},
 'enable_memory': True,
 'resources': {'cpu': '1000m',
               'engine': 'py

### 5.7 List Deployments

Lists all current deployments so you can see what is running or pending in the environment.

In [ ]:
show_output("List Deployments", deployments.list())


### 5.8 Get Deployment Logs

Retrieves the last 200 lines of logs from the running deployment.

In [ ]:
logs = deployments.logs(id=DEPLOYMENT_ID, tail_lines=200, container=None, follow=False)
show_output("Deployment Logs (last 200 lines)", logs)

<a id="agent-invocation"></a>
## 6. Validate the Agent

Calls the deployed agent directly to confirm it is healthy and responding: `/health`, `/info`, and `/invoke`.


### 6.1 Get Agent URL

Retrieves the deployment details and captures `agent_url` for the runtime tests in Section 6.2.

In [144]:
deployment = deployments.get(id=DEPLOYMENT_ID)
agent_url = getattr(deployment, "agent_url", None)
dep_name = getattr(deployment, "deployment_name", None)
dep_status = getattr(deployment, "status", None)

print(f"Deployment : {dep_name}")
print(f"Agent ID   : {DEPLOYMENT_ID}")
print(f"Status     : {dep_status}")
print(f"Agent URL  : {agent_url}")

Deployment : LangGraph-Agent-test-2-new-test
Agent ID   : <uuid>
Status     : active
Agent URL  : http://<service-host>/one-td/agents/langgraph-agent-test-2-new-ae64c4fd


### 6.2 Test Agent Endpoints

Tests the deployed agent's `/health`, `/info`, and `/invoke` endpoints. Each request prints the HTTP status code and the response body.

In [ ]:
if not agent_url:
    raise RuntimeError(
        "agent_url is not set. Run cell 6.1 (Get Agent URL) first, and make sure the "
        "deployment is active (cell 5.5). If agent_url is still empty afterwards, the "
        "deployment may not expose a URL yet — check its status with cell 5.6."
    )

headers = {"Authorization": f"Bearer {AUTH_TOKEN}"}

print("=" * 80)
print(f"Agent URL: {agent_url}")
print("=" * 80)

# /health
response = requests.get(f"{agent_url}/health", headers=headers, timeout=30, verify=SSL_VERIFY)
print(f"\n/health  → {response.status_code}")
payload = response.json() if response.ok else response.text
print(json.dumps(payload, indent=2) if response.ok else payload)

# /info
response = requests.get(f"{agent_url}/info", headers=headers, timeout=30, verify=SSL_VERIFY)
print(f"\n/info    → {response.status_code}")
payload = response.json() if response.ok else response.text
print(json.dumps(payload, indent=2) if response.ok else payload)

# /invoke
invoke_payload = {
    "input": {"messages": [{"role": "user", "content": "Hello, are you working?"}]},
    "config": {},
}
response = requests.post(f"{agent_url}/invoke", json=invoke_payload, headers=headers, timeout=30, verify=SSL_VERIFY)
print(f"\n/invoke  → {response.status_code}")
payload = response.json() if response.ok else response.text
print(json.dumps(payload, indent=2) if response.ok else payload)

### 6.3 Restart the Deployment *(optional)*

Run this cell to perform a rolling restart of the pods, then wait briefly and show the post-restart status. Skip it if you don't need a restart.


In [146]:
result = deployments.restart(id=DEPLOYMENT_ID)
show_output("Restart Agent", result)

print("\nWaiting for pods to restart (30 seconds)...")
time.sleep(30)
show_output("Post-restart Status", deployments.get(id=DEPLOYMENT_ID))



Restart Agent:
{'deployment_id': '<uuid>',
 'status': 'active',
 'message': 'Rolling restart initiated at 2026-06-18T14:40:04Z',
 'deployment_name': 'langgraph-agent-test-2-new-ae64c4fd',
 'namespace': 'agentruntime',
 'replicas': 1}

Waiting for pods to restart (30 seconds)...

Post-restart Status:
{'in_house_observability': True,
 'agent_id': '<uuid>',
 'agent_definition_id': '<uuid>',
 'agent_definition_name': 'langgraph-agent-demo-new-test',
 'workspace_id': '<uuid>',
 'deployment_name': 'LangGraph-Agent-test-2-new-test',
 'environment': 'dev',
 'namespace': 'agentruntime',
 'version_id': '<uuid>',
 'version_number': 1,
 'deployment_history': [{'user': '<user>',
                         'record_id': '<uuid>',
                         'timestamp': '2026-06-18T14:38:47.911245+00:00',
                         'version_id': '<uuid>',
                         'previous_version_id': None}],
 'runtime_config': {'OPENAI_API_KEY': '<redacted-secret>',
                    'OPIK_PROJECT_NAME

<a id="upgrade"></a>
## 7. Version Management *(optional)*

These steps show how to evolve a running agent: update metadata, pull new code from Git, inspect and list versions, and upgrade the deployment. The metadata and read-only cells are safe to run as-is; the Git pull (7.2) and deployment upgrade (7.5) prompt for the extra input they need when you run them.


### 7.1 Update Agent Definition Metadata

Updates the name, description, or tags of the active agent definition without creating a new deployment.

In [147]:
updated_def = agent_definitions.update(
    id=AGENT_DEF_ID,
    body=AgentDefinitionUpdateRequest(
        name=AGENT_DEFINITION_NAME,
        description="Enhanced LangGraph agent with improved capabilities",
        tags=AGENT_TAGS + ["updated"],
    ),
)
show_output("Updated Agent Definition", updated_def)



Updated Agent Definition:
{'id': '<uuid>',
 'message': "Agent 'langgraph-agent-demo-new-test' updated successfully",
 'name': 'langgraph-agent-demo-new-test',
 'status': 'published',
 'updated_at': '2026-06-18T14:40:45.060469Z'}


### 7.2 Pull the Latest Git Version *(optional)*

Run this cell only if your agent definition was created from Git (cell 4.2). It reuses the Git credentials you already entered there, pulls the latest code, and registers it as a **new version** (for example, v2). Use the List versions cell (7.4) to confirm the new identifier before you upgrade or archive.


In [148]:
# Reuse the Git credentials entered in the Git create cell (4.2).
if "git_credentials" not in globals():
    print("No Git credentials found. Run the Git create cell (4.2) first if your definition is Git-based.")
else:
    sync_result = agent_definitions.pull_latest_version(
        id=AGENT_DEF_ID,
        body=GitSyncRequest(
            branch=GIT_BRANCH,
            target_path=GIT_TARGET_PATH or None,
            version_notes="Pulled latest changes from Git",
            credentials=git_credentials,
        ),
    )
    show_output("Pull Latest Version from Git", sync_result)



Pull Latest Version from Git:
{'changed': False,
 'message': "Already up to date. Matching version 1 already contains commit 'a896a31aa4a60b5589f11397d2d983744d550380' "
            "from main branch, and target path 'examples/agents/sample_agent'.",
 'current_version': {'version_id': '<uuid>', 'version_number': 1},
 'new_version': None,
 'previous_commit': None,
 'new_commit': None}


### 7.3 Get Version Details

Retrieves the status and provenance of the version set by `GET_VERSION_IDENTIFIER` at the top of this cell. Change that value to inspect a different version.


In [149]:
GET_VERSION_IDENTIFIER = "1"  # version to inspect

version_detail = agent_definitions.get_version(id=AGENT_DEF_ID, version_identifier=GET_VERSION_IDENTIFIER)
show_output("Get Version", version_detail)



Get Version:
{'id': '<uuid>',
 'agent_definition_id': '<uuid>',
 'version_number': 1,
 'status': 'published',
 'version_notes': None,
 'artifacts_path': '<uuid>/agent-artifacts/<uuid>/v1',
 'error_message': None,
 'created_at': '2026-06-18T14:37:43.234291Z',
 'updated_at': '2026-06-18T14:37:45.139125Z',
 'code_artifact_url': 's3://<internal-host>-one-td-agentops-artifacts/<uuid>/agent-artifacts/<uuid>/v1/',
 'config_artifact_url': 's3://<internal-host>-one-td-agentops-artifacts/<uuid>/agent-artifacts/<uuid>/v1/',
 'source_commit_sha': 'a896a31aa4a60b5589f11397d2d983744d550380',
 'source_repository_url': 'https://github.com/<org>/<repo>.git',
 'source_branch': 'main',
 'source_target_path': 'examples/agents/sample_agent',
 'default_config': {'environment_variables': {'OPENAI_API_KEY': '<redacted-secret>',
                                              'OPENAI_MODEL_NAME': 'us-anthropic-claude-sonnet-4-5-20250929-v1-0-profile',
                                              'TD_AI_STUDIO_

### 7.4 List Agent Definition Versions

Lists all versions for the active agent definition. Use the output to choose the version identifier for deployment update or version archive.

In [150]:
vers = agent_definitions.list_versions(
    id=AGENT_DEF_ID,
    status=None,
    include_archived=False,
    include_failed=False,
    limit=50,
    offset=0,
)
show_output("List Versions", vers)


List Versions:
{'versions': [{'id': '<uuid>',
               'agent_definition_id': '<uuid>',
               'version_number': 1,
               'status': 'published',
               'version_notes': None,
               'artifacts_path': '<uuid>/agent-artifacts/<uuid>/v1',
               'error_message': None,
               'created_at': '2026-06-18T14:37:43.234291Z',
               'updated_at': '2026-06-18T14:37:45.139125Z'}],
 'total': 1,
 'skip': 0,
 'limit': 50,
 'agent_definition_id': '<uuid>',
 'agent_name': 'langgraph-agent-demo-new-test'}


### 7.5 Upgrade the Deployment *(optional)*

Run this cell to roll the running deployment onto the version set by `UPGRADE_VERSION_IDENTIFIER` at the top of this cell. List the versions above first, then adjust that value if needed.


In [151]:
UPGRADE_VERSION_IDENTIFIER = "1"  # version to deploy

result = deployments.update(
    id=DEPLOYMENT_ID,
    body=DeploymentUpdateRequest(
        agent_definition_id=AGENT_DEF_ID,
        version_identifier=UPGRADE_VERSION_IDENTIFIER,
    ),
)
show_output("Update Deployment", result)
print(f"Deployment updated to version {UPGRADE_VERSION_IDENTIFIER}")



Update Deployment:
{'in_house_observability': True,
 'agent_id': '<uuid>',
 'agent_definition_id': '<uuid>',
 'agent_definition_name': 'langgraph-agent-demo-new-test',
 'workspace_id': '<uuid>',
 'deployment_name': 'LangGraph-Agent-test-2-new-test',
 'environment': 'dev',
 'namespace': 'agentruntime',
 'version_id': '<uuid>',
 'version_number': 1,
 'deployment_history': [{'user': '<user>',
                         'record_id': '<uuid>',
                         'timestamp': '2026-06-18T14:40:59.991208+00:00',
                         'version_id': '<uuid>',
                         'previous_version_id': '<uuid>'},
                        {'user': '<user>',
                         'record_id': '<uuid>',
                         'timestamp': '2026-06-18T14:38:47.911245+00:00',
                         'version_id': '<uuid>',
                         'previous_version_id': None}],
 'runtime_config': {'OPENAI_API_KEY': '<redacted-secret>',
                    'OPIK_PROJECT_NAME': '<uuid

### 7.6 Monitor Upgrade Progress

Polls the deployment after an upgrade to confirm when the rollout completes.

In [152]:
deployments.poll(id=DEPLOYMENT_ID)

Polling Deployments (every 10s, max 30 attempts)...
  [1/30] Status: active
    current_step: Deployment active and healthy
    steps_completed: pending, building, pushing, deploying, health_check, active
  Done — status: active


DeploymentResponse(in_house_observability=True, agent_id='<uuid>', agent_definition_id='<uuid>', agent_definition_name='langgraph-agent-demo-new-test', workspace_id='<uuid>', deployment_name='LangGraph-Agent-test-2-new-test', environment='dev', namespace='agentruntime', version_id='<uuid>', version_number=1, deployment_history=[{'user': '<user>', 'record_id': '<uuid>', 'timestamp': '2026-06-18T14:40:59.991208+00:00', 'version_id': '<uuid>', 'previous_version_id': '<uuid>'}, {'user': '<user>', 'record_id': '<uuid>', 'timestamp': '2026-06-18T14:38:47.911245+00:00', 'version_id': '<uuid>', 'previous_version_id': None}], runtime_config={'OPENAI_API_KEY': '<redacted-secret>', 'OPIK_PROJECT_NAME': '<uuid>', 'OPIK_URL_OVERRIDE': 'http://<internal-host>:8080', 'ENFORCE_AUTHORIZATION': 'true', 'OPIK_CONSOLE_LOGGING_LEVEL': 'DEBUG'}, enable_memory=True, resources={'cpu': '1000m', 'engine': 'python', 'memory': '2Gi', 'service': {'port': 5000, 'type': 'ClusterIP', 'target_port': 5000}, 'replicas':

<a id="cleanup"></a>
## 8. Cleanup

Removes all resources created during this walkthrough. Run these sections in order.

### 8.1 Delete Deployment

Deletes the running deployment and removes its Kubernetes workload from the environment.

In [153]:
if DEPLOYMENT_ID:
    show_output("Delete Deployment", deployments.delete(id=DEPLOYMENT_ID))
else:
    print("No DEPLOYMENT_ID set — nothing to delete.")



Delete Deployment:
{'deployment_id': '<uuid>',
 'status': 'deleted',
 'message': 'Deployment deleted successfully',
 'deployment_name': 'langgraph-agent-test-2-new-ae64c4fd',
 'namespace': 'agentruntime'}


### 8.2 Delete Secrets

Deletes the runtime and image pull secrets created by this notebook. Secrets with no ID (never created) are skipped automatically.

In [154]:
secret_ids_to_delete = [
    (SECRET_ID_1, "runtime secret"),
    (SECRET_ID_2, "image pull secret"),
]

for secret_id, label in secret_ids_to_delete:
    if secret_id:
        show_output(
            f"Delete {label} ({secret_id})",
            secrets.delete(id=secret_id, workspace_id=WORKSPACE_ID, force=False),
        )
    else:
        print(f"Skipping {label}: not created.")



Delete runtime secret (<uuid>):
''

Delete image pull secret (<uuid>):
''


### 8.3 Archive an Agent Definition Version *(optional)*

Run this cell only if you want to archive a single version (rather than the whole definition). It archives the version set by `ARCHIVE_VERSION_IDENTIFIER` at the top of this cell.


In [155]:
ARCHIVE_VERSION_IDENTIFIER = "1"  # version to archive

try:
    show_output(
        "Archive Version",
        agent_definitions.archive_version(
            id=AGENT_DEF_ID,
            body=AgentDefinitionVersionArchiveRequest(version_identifier=ARCHIVE_VERSION_IDENTIFIER),
        ),
    )
except Exception as exc:
    if "Version already archived" in str(exc) or "CONFLICT_ERROR" in str(exc):
        print(f"Version {ARCHIVE_VERSION_IDENTIFIER} is already archived. No action needed.")
    else:
        raise


Version 1 is already archived. No action needed.


### 8.4 Archive the Agent Definition

Archives the agent definition itself. After archiving, no new deployments can reference this definition.

In [156]:
if AGENT_DEF_ID:
    show_output("Archive Agent Definition", agent_definitions.archive(id=AGENT_DEF_ID))
else:
    print("No AGENT_DEF_ID set — nothing to archive.")



Archive Agent Definition:
{'id': '<uuid>', 'message': "Agent 'langgraph-agent-demo-new-test' archived"}


<a id="data-models"></a>
## 9. Data Models Reference

### 9.1 Agent Definition Status

| Status | Description |
|--------|-------------|
| `creating` | Initial state, artifacts being processed in background |
| `published` | Ready for deployment |
| `failed` | Artifact processing failed |
| `archived` | Soft deleted, not available for new deployments |

### 9.2 Agent Definition Version Status

| Status | Description |
|--------|-------------|
| `creating` | Version being created, artifacts being processed |
| `published` | Version ready for deployment |
| `failed` | Version creation or processing failed |
| `archived` | Version soft deleted, not available for new deployments |

**Note:** Version numbers are auto-incremented (v1, v2, v3...). Uploading identical artifacts returns the existing version.

### 9.3 Deployment Status Lifecycle

| Status | Description | Next Status |
|--------|-------------|-------------|
| `pending` | Deployment queued | `building` or `failed` |
| `building` | Packaging agent artifacts | `pushing` or `failed` |
| `pushing` | Uploading package to S3 | `deploying` or `failed` |
| `deploying` | Creating Kubernetes resources | `health_check` or `failed` |
| `health_check` | Waiting for pods to be ready | `active` or `failed` |
| `active` | Deployment running successfully | `stopped`, `deleted` |
| `stopped` | Scaled to 0 replicas (via stop endpoint) | `active`, `deleted` |
| `failed` | Error occurred (check `error_message` field) | Terminal state |
| `deleted` | Removed from Kubernetes | Terminal state |

### 9.4 Artifact Storage Types

| Type | Required Fields | Description |
|------|-----------------|-------------|
| `zip` | `file` (multipart upload) | Local ZIP file upload |
| `git` | `git_config.repository_url` | Git repository clone |
| `s3` | `s3_config.bucket_name`, `s3_config.object_path` | AWS S3 / MinIO storage |
| `azure_blob` | `azure_blob_config.container_name`, `azure_blob_config.blob_path` | Azure Blob Storage |

### 9.5 Git Configuration Fields (`ArtifactsConfig.git_config`)

| Field | Required | Default | Description |
|-------|----------|---------|-------------|
| `repository_url` | Yes | - | HTTPS or SSH Git URL |
| `branch` | No | `"main"` | Branch to clone |
| `target_path` | No | `null` | Subdirectory for monorepos |
| `credentials` | No | `null` | `GitCredentials` (PAT or username/password) |

### 9.6 Secret Types

| Type | Model | Description |
|------|-------|-------------|
| `opaque` | `OpaqueSecretData` | Generic key-value secret |
| `tls` | `TLSSecretData` | TLS certificate |
| `dockerconfigjson` | `DockerConfigJsonSecretData` | Docker registry authentication |


Last validated: 2026-06-19
